In [2]:
#import the pdf:
# for local file
from magic_doc.docconv import DocConverter
converter = DocConverter(s3_config=None)
markdown_content, time_cost = converter.convert("/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/erc-B1.pdf", conv_timeout=300)

#print(markdown_content)


2024-07-26 20:49:01.935 | INFO     | magic_pdf.libs.pdf_check:detect_invalid_chars:57 - cid_count: 0, text_len: 25538, cid_chars_radio: 0.0
2024-07-26 20:49:01.958 | INFO     | magic_doc.contrib.pdf.pdf_extractor:run:70 - stream io data is digital pdf


In [3]:
import ollama
from nltk import word_tokenize
import re
import math

# Define the refined personalities and their corresponding parameters
personalities_parameters = {
    'Highly analytical evaluator': {
        'model': 'llama3.1',
        'temperature': 0.2,
        'top_p': 0.8,
        'frequency_penalty': 1.5,
        'presence_penalty': 1.2
    },
    'Collaboration expert': {
        'model': 'llama3.1',
        'temperature': 0.2,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    },
    'Innovation and impact specialist': {
        'model': 'llama3.1',
        'temperature': 0.2,
        'top_p': 0.95,
        'frequency_penalty': 0.8,
        'presence_penalty': 1.0
    },
    'Project management expert': {
        'model': 'llama3.1',
        'temperature': 0.2,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    }
}

# Define the section-specific and complex questions
questions = {
    "Extended Synopsis": {
        'Highly analytical evaluator': "Does the Extended Synopsis present clear, measurable, and verifiable objectives? Is the proposed methodology sound, with all underlying concepts, models, and assumptions well-explained and logically consistent? Does it clearly highlight the ground-breaking nature of the research project and the feasibility of the outlined scientific approach?",
        'Collaboration expert': "Does the Extended Synopsis demonstrate effective engagement with all relevant stakeholders, including examples of successful past collaborations or plans for future partnerships? Does it involve appropriate interdisciplinary approaches?",
        'Innovation and impact specialist': "Does the Extended Synopsis clearly highlight how the project goes beyond the state-of-the-art with innovative concepts, approaches, and methodologies? Are the expected outcomes and impacts clearly described, with an emphasis on innovation and the generation of new opportunities?",
        'Project management expert': "Does the Extended Synopsis provide a detailed and efficient structure, including clear objectives, timelines, and resource allocation, with well-defined milestones and deliverables that ensure effective implementation?"
    },
    "Curriculum Vitae and Track Record": {
        'Highly analytical evaluator': "Does the Curriculum Vitae and Track Record provide a comprehensive overview of the Principal Investigator's education, key qualifications, and relevant previous positions? Are the listed research outputs and peer recognition examples well-justified and demonstrate the applicant's capacity to successfully carry out the proposed project?",
        'Collaboration expert': "Does the Curriculum Vitae and Track Record include relevant additional information on career breaks, unconventional career paths, and life events? Does it highlight any particularly noteworthy contributions to the research community beyond research achievements and peer recognition?",
        'Innovation and impact specialist': "Does the Curriculum Vitae and Track Record demonstrate the applicant's ability to advance knowledge in their field, with an emphasis on more recent achievements? Are the listed research outputs and peer recognition examples innovative and impactful?",
        'Project management expert': "Does the Curriculum Vitae and Track Record provide a clear and structured overview of the Principal Investigator's career and achievements? Are the listed research outputs and peer recognition examples well-aligned with the proposed project's objectives and timelines?"
    }
}

def calculate_context_window(sections):
    section_word_counts = [len(word_tokenize(section)) for section in sections]
    max_word_count = max(section_word_counts)
    context_window = int(max_word_count * 1.2)
    context_window = math.ceil(context_window / 10000) * 10000
    return context_window

def analyze_section(personality_key, params, section_title, section, context_window):
    question = questions[section_title][personality_key]
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are an helpful assistant. You must behave and act as a {personality_key}, and you are also an expert evaluating an EU grant application. '
                'Your audience includes researchers and research support officers. You do not have access to internet resources, and your analysis should be based solely on the provided document. '
                'The document consists of two main sections: "Extended Synopsis" and "Curriculum Vitae and Track Record." '
                f'Use the following section of the document, titled "{section_title}", to answer the question. '
                'Follow these steps in your response:\n'
                '1. Provide a detailed analysis, including specific strengths and weaknesses of the section. List each strength and weakness as a separate bullet point and include as many as you find relevant.\n'
                '2. Offer actionable recommendations for improvement, with each recommendation as a separate bullet point.\n'
                '3. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Document: {section}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': question,
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": params['temperature'],
        "top_p": params['top_p'],
        "frequency_penalty": params['frequency_penalty'],
        "presence_penalty": params['presence_penalty']
    }
    response = ollama.chat(model=params['model'], messages=messages, options=options)
    response_content = response['message']['content']
    return response_content

def evaluate_all_experts(document_text):
    sections = {
        "Extended Synopsis": document_text["Extended Synopsis"],
        "Curriculum Vitae and Track Record": document_text["Curriculum Vitae and Track Record"]
    }
    context_window = calculate_context_window(sections.values())

    all_expert_responses = []

    for personality_key, params in personalities_parameters.items():
        expert_responses = {"personality": personality_key, "responses": {}}
        for section_title, section in sections.items():
            print(f"The {personality_key} is analyzing the {section_title} section")
            answer = analyze_section(personality_key, params, section_title, section, context_window)
            expert_responses["responses"][section_title] = answer
        all_expert_responses.append(expert_responses)

    return all_expert_responses

def analyze_reviewer(section_title, consolidated_feedback):
    context_window = 5000
    messages = [
        {
            'role': 'system',
            'content': (
                'You are Comprehensive Reviewer, an expert tasked with synthesizing feedback from multiple expert reviews of an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of two main sections: "Extended Synopsis" and "Curriculum Vitae and Track Record." '
                f'Use the consolidated feedback for the section titled "{section_title}" to provide a synthesis. '
                'Follow these steps in your response:\n'
                '1. Summarize the key strengths identified by the experts.\n'
                '2. Summarize the key weaknesses identified by the experts.\n'
                '3. Provide actionable recommendations for improvement based on the experts’ feedback.\n'
                '4. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Consolidated Feedback: {consolidated_feedback[section_title]}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': (
                f'Please provide a comprehensive synthesis for the section titled "{section_title}" based on the consolidated feedback provided.'
            ),
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": 0.5,
        "top_p": 0.9,
        "frequency_penalty": 1.0,
        "presence_penalty": 1.0
    }
    response = ollama.chat(model="llama3.1", messages=messages, options=options)
    response_content = response['message']['content']
    return response_content

def flatten_responses(all_expert_responses):
    flat_expert_responces = ""

    for expert in all_expert_responses:
        flat_expert_responces += f"Personality: {expert['personality']}\n\n"
        for section, response in expert["responses"].items():
            flat_expert_responces += f"Section: {section}\n"
            flat_expert_responces += f"Response:\n{response}\n\n"
        flat_expert_responces += "="*50 + "\n\n"

    return flat_expert_responces

def combine_reviews(final_review, combined_review):
    # Combine the reviews into a single string
    combined_content = []

    combined_content.append("# Final Review and Combined Review\n\n")

    # Append final review content
    combined_content.append("## Final Review\n")
    combined_content.append(final_review)
    combined_content.append("\n\n")

    # Append combined review content
    combined_content.append("## Combined Review\n")
    combined_content.append(combined_review)
    combined_content.append("\n")

    return "\n".join(combined_content)

def save_to_markdown(content, filepath):
    # Save the combined content to a markdown file
    with open(filepath, "w", encoding="utf-8") as file:
        file.write(content)

# Example document content
document_text = {
    "Extended Synopsis": "Your extended synopsis text here...",
    "Curriculum Vitae and Track Record": "Your CV and track record text here..."
}

all_expert_responses = evaluate_all_experts(document_text)

# Example usage of flatten_responses and save_to_markdown
flat_responses = flatten_responses(all_expert_responses)
save_to_markdown(flat_responses, "expert_responses.md")


The Highly analytical evaluator is analyzing the Extended Synopsis section
The Highly analytical evaluator is analyzing the Curriculum Vitae and Track Record section
The Collaboration expert is analyzing the Extended Synopsis section
The Collaboration expert is analyzing the Curriculum Vitae and Track Record section
The Innovation and impact specialist is analyzing the Extended Synopsis section
The Innovation and impact specialist is analyzing the Curriculum Vitae and Track Record section
The Project management expert is analyzing the Extended Synopsis section
The Project management expert is analyzing the Curriculum Vitae and Track Record section


In [4]:
# print the expert responses in markdown format
print(flat_responses)


Personality: Highly analytical evaluator

Section: Extended Synopsis
Response:
**Section: Extended Synopsis**

1. **Strengths**:
   * The proposal presents a clear problem statement related to [specific area], which is relevant to EU priorities.
   * Objectives are well-defined, specific, measurable (e.g., "improve efficiency by 20%"), achievable ("within the next two years"), Relevant ("to European industry and society") and Time-bound ("by project completion").
   * The proposed methodology seems sound with a logical approach outlined for data collection and analysis.
   * A clear timeline is presented indicating milestones, which helps in understanding how objectives will be met within specified timelines.

2. **Weaknesses**:
   - While the problem statement is relevant to EU priorities, it could benefit from more specific details on why this issue affects Europe uniquely or has significant implications for European society.
   - The proposal does not clearly differentiate itself as